In [2]:
# ============================================================
# XGBOOST - OPTIMIZACIÓN DEL THRESHOLD
# ============================================================
#
# NO SE UTILIZA EL TEST OFICIAL.
#
# Configuración congelada:
#
# MX -> normal
# ES -> lemma
# CU -> stem
#
# Representación:
# Word TF-IDF (1,2) + 9 features lingüísticas
#
# Modelo:
# XGBoost con hiperparámetros optimizados previamente
#
# Objetivo:
# Buscar el threshold que maximiza F1-Macro
# mediante predicciones Out-Of-Fold (OOF).
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import json

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score
)

from xgboost import XGBClassifier


# ============================================================
# 2. CONFIGURACIÓN GENERAL
# ============================================================

RANDOM_STATE = 42

DATA_DIR = '../data'

VARIANTES = [
    'mx',
    'es',
    'cu'
]


FEATURE_COLS = [
    'n_exc',
    'n_int',
    'n_may',
    'n_emo',
    'n_ris',
    'n_neg',
    'n_elo',
    'n_com',
    'n_pun'
]


CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 3. PREPROCESSING GANADOR
# ============================================================

MEJOR_PREP = {

    'mx': 'normal',

    'es': 'lemma',

    'cu': 'stem'
}


PREPROCESAMIENTOS = {

    'normal': '',

    'stem': '_stem',

    'lemma': '_lemma'
}


# ============================================================
# 4. MEJORES HIPERPARÁMETROS DE XGBOOST
# ============================================================

BEST_PARAMS = {

    # --------------------------------------------------------
    # MÉXICO - PARÁMETROS EXACTOS
    # --------------------------------------------------------

    'mx': {

        'n_estimators': 475,

        'max_depth': 6,

        'learning_rate':
            0.021061679900335632,

        'min_child_weight': 3,

        'subsample':
            0.6517559963200034,

        'colsample_bytree':
            0.7825044568707964,

        'gamma':
            2.9351332282682328,

        'reg_alpha':
            0.0036324869566766076,

        'reg_lambda':
            0.5143828405076928,

        'scale_pos_weight':
            1.529398118867869
    },


    # --------------------------------------------------------
    # ESPAÑA - PARÁMETROS EXACTOS
    # --------------------------------------------------------

    'es': {

        'n_estimators': 775,

        'max_depth': 4,

        'learning_rate':
            0.029541163614874227,

        'min_child_weight': 1,

        'subsample':
            0.9585995985547178,

        'colsample_bytree':
            0.8836941661004954,

        'gamma':
            2.8417377011176943,

        'reg_alpha':
            0.04119379042044028,

        'reg_lambda':
            6.867346536918053,

        'scale_pos_weight':
            1.270315808816275
    },


    # --------------------------------------------------------
    # CUBA
    #
    # Estos son los parámetros REDONDEADOS que conservamos.
    #
    # Cuando recuperemos los exactos,
    # únicamente reemplazamos este bloque.
    # --------------------------------------------------------

    'cu': {
        'n_estimators': 450,
        'max_depth': 3,
        'learning_rate': 0.03169263099895088,
        'min_child_weight': 1,
        'subsample': 0.7816524076750944,
        'colsample_bytree': 0.7950660358381656,
        'gamma': 3.706704516605376,
        'reg_alpha': 0.004749825692106681,
        'reg_lambda': 0.35253189292857723,
        'scale_pos_weight': 1.515053442659512
    }
}


# ============================================================
# 5. F1-CV DE REFERENCIA
# ============================================================
#
# Sirve solamente para verificar que estamos aproximadamente
# reproduciendo los modelos anteriores.
#
# En CU puede existir diferencia porque los parámetros
# disponibles están redondeados.
#
# ============================================================

F1_REFERENCIA = {

    'mx': 0.6259,

    'es': 0.7135,

    'cu': 0.6743
}


# ============================================================
# 6. CARGAR TRAIN
# ============================================================

def cargar_train(variante):

    prep = MEJOR_PREP[variante]

    sufijo = PREPROCESAMIENTOS[prep]

    ruta = (
        f'{DATA_DIR}/'
        f'train_clean{sufijo}_{variante}.csv'
    )

    print(
        f'Cargando: {ruta}'
    )

    df = pd.read_csv(
        ruta
    )

    return df


# ============================================================
# 7. TF-IDF + 9 FEATURES
# ============================================================
#
# Esta es la representación que ganó:
#
# Word TF-IDF
# ngram_range = (1,2)
# max_features = 20 000
# min_df = 2
# max_df = 0.98
# sublinear_tf = True
#
# ============================================================

def crear_preprocesador():

    tfidf = TfidfVectorizer(

        analyzer='word',

        ngram_range=(1, 2),

        min_df=2,

        max_df=0.98,

        max_features=20000,

        sublinear_tf=True,

        lowercase=False,

        dtype=np.float32
    )


    preprocesador = ColumnTransformer(

        transformers=[

            (
                'tfidf',
                tfidf,
                'MESSAGE_CLEAN'
            ),

            (
                'linguisticas',
                'passthrough',
                FEATURE_COLS
            )
        ],

        remainder='drop'
    )


    return preprocesador


# ============================================================
# 8. CREAR XGBOOST
# ============================================================

def crear_xgboost(variante):

    params = (
        BEST_PARAMS[variante]
        .copy()
    )


    modelo = XGBClassifier(

        objective='binary:logistic',

        tree_method='hist',

        eval_metric='logloss',

        random_state=RANDOM_STATE,

        n_jobs=-1,

        **params
    )


    return modelo


# ============================================================
# 9. CREAR PIPELINE
# ============================================================

def crear_pipeline(variante):

    pipeline = Pipeline(

        steps=[

            (
                'features',
                crear_preprocesador()
            ),

            (
                'xgb',
                crear_xgboost(
                    variante
                )
            )
        ]
    )


    return pipeline


# ============================================================
# 10. OBTENER PREDICCIONES OUT-OF-FOLD
# ============================================================
#
# Cada texto recibe una probabilidad calculada por un modelo
# que NO fue entrenado con ese texto.
#
# Esto permite buscar el threshold sin tocar el test.
#
# ============================================================

def obtener_oof(variante):

    df = cargar_train(
        variante
    )


    columnas_X = (
        ['MESSAGE_CLEAN']
        + FEATURE_COLS
    )


    X = df[
        columnas_X
    ].copy()


    y = (
        df['IS_IRONIC']
        .astype(int)
    )


    # --------------------------------------------------------
    # NaN
    # --------------------------------------------------------

    X['MESSAGE_CLEAN'] = (

        X['MESSAGE_CLEAN']
        .fillna('')
        .astype(str)
    )


    X[FEATURE_COLS] = (

        X[FEATURE_COLS]
        .fillna(0)
    )


    pipeline = crear_pipeline(
        variante
    )


    # --------------------------------------------------------
    # Predicciones OOF
    #
    # predict_proba devuelve:
    #
    # [:,0] -> prob clase 0
    # [:,1] -> prob clase 1
    #
    # --------------------------------------------------------

    probabilidades = cross_val_predict(

        estimator=pipeline,

        X=X,

        y=y,

        cv=CV,

        method='predict_proba',

        n_jobs=1
    )


    prob_clase_1 = (
        probabilidades[:, 1]
    )


    return (
        y.to_numpy(),
        prob_clase_1
    )


# ============================================================
# 11. BUSCAR MEJOR THRESHOLD
# ============================================================
#
# Primero hacemos una búsqueda fina:
#
# 0.20 -> 0.80
# paso 0.005
#
# ============================================================

def buscar_threshold(
    y_true,
    probabilidades
):

    thresholds = np.arange(
        0.20,
        0.801,
        0.005
    )


    resultados = []


    for threshold in thresholds:

        y_pred = (
            probabilidades
            >= threshold
        ).astype(int)


        f1_macro = f1_score(
            y_true,
            y_pred,
            average='macro'
        )


        accuracy = accuracy_score(
            y_true,
            y_pred
        )


        precision = precision_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0
        )


        recall = recall_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0
        )


        resultados.append({

            'threshold':
                threshold,

            'f1_macro':
                f1_macro,

            'accuracy':
                accuracy,

            'precision_macro':
                precision,

            'recall_macro':
                recall
        })


    df_thresholds = pd.DataFrame(
        resultados
    )


    df_thresholds = (

        df_thresholds
        .sort_values(
            'f1_macro',
            ascending=False
        )
        .reset_index(drop=True)
    )


    return df_thresholds


# ============================================================
# 12. EJECUTAR PARA LAS TRES VARIANTES
# ============================================================

resultados_threshold = []

historial_thresholds = {}

oof_probabilidades = {}


for variante in VARIANTES:

    print('\n')

    print('=' * 70)

    print(
        f'THRESHOLD TUNING - '
        f'{variante.upper()}'
    )

    print(
        f'Preprocessing: '
        f'{MEJOR_PREP[variante]}'
    )

    print('=' * 70)


    # --------------------------------------------------------
    # OOF
    # --------------------------------------------------------

    (
        y_true,
        probabilidades

    ) = obtener_oof(
        variante
    )


    oof_probabilidades[
        variante
    ] = {

        'y_true':
            y_true,

        'probabilidades':
            probabilidades
    }


    # --------------------------------------------------------
    # F1 CON THRESHOLD 0.5
    # --------------------------------------------------------

    y_pred_05 = (
        probabilidades
        >= 0.50
    ).astype(int)


    f1_05 = f1_score(

        y_true,

        y_pred_05,

        average='macro'
    )


    # --------------------------------------------------------
    # Buscar threshold
    # --------------------------------------------------------

    df_threshold = buscar_threshold(

        y_true,

        probabilidades
    )


    historial_thresholds[
        variante
    ] = df_threshold


    mejor = (
        df_threshold
        .iloc[0]
    )


    mejor_threshold = (
        mejor['threshold']
    )


    mejor_f1 = (
        mejor['f1_macro']
    )


    mejora = (
        mejor_f1
        - f1_05
    )


    resultados_threshold.append({

        'variante':
            variante,

        'preprocesamiento':
            MEJOR_PREP[variante],

        'threshold_base':
            0.50,

        'f1_threshold_05':
            f1_05,

        'threshold_optimo':
            mejor_threshold,

        'f1_threshold_optimo':
            mejor_f1,

        'mejora':
            mejora,

        'f1_referencia_anterior':
            F1_REFERENCIA[variante]
    })


    print(
        f'\nF1-Macro threshold 0.50: '
        f'{f1_05:.4f}'
    )


    print(
        f'Threshold óptimo: '
        f'{mejor_threshold:.3f}'
    )


    print(
        f'F1-Macro óptimo: '
        f'{mejor_f1:.4f}'
    )


    print(
        f'Mejora: '
        f'{mejora:+.4f}'
    )


# ============================================================
# 13. RESULTADO GENERAL
# ============================================================

df_threshold_final = pd.DataFrame(
    resultados_threshold
)


print('\n')

print('=' * 75)

print(
    'RESULTADOS FINALES '
    'DE THRESHOLD TUNING'
)

print('=' * 75)


display(

    df_threshold_final
    .style
    .format({

        'threshold_base':
            '{:.3f}',

        'f1_threshold_05':
            '{:.4f}',

        'threshold_optimo':
            '{:.3f}',

        'f1_threshold_optimo':
            '{:.4f}',

        'mejora':
            '{:+.4f}',

        'f1_referencia_anterior':
            '{:.4f}'
    })
)


# ============================================================
# 14. TOP 10 THRESHOLDS POR VARIANTE
# ============================================================

for variante in VARIANTES:

    print('\n')

    print('=' * 70)

    print(
        f'TOP 10 THRESHOLDS - '
        f'{variante.upper()}'
    )

    print('=' * 70)


    display(

        historial_thresholds[
            variante
        ]
        .head(10)
        .style
        .format({

            'threshold':
                '{:.3f}',

            'f1_macro':
                '{:.4f}',

            'accuracy':
                '{:.4f}',

            'precision_macro':
                '{:.4f}',

            'recall_macro':
                '{:.4f}'
        })
    )


# ============================================================
# 15. GUARDAR THRESHOLDS GANADORES
# ============================================================

BEST_THRESHOLDS = {

    fila['variante']:
        float(
            fila['threshold_optimo']
        )

    for fila
    in resultados_threshold
}


print('\n')

print('=' * 70)

print(
    'THRESHOLDS DEFINITIVOS'
)

print('=' * 70)

print(
    BEST_THRESHOLDS
)


# ============================================================
# 16. GUARDAR EN JSON
# ============================================================
#
# Así no perdemos los resultados si VS Code se cierra.
#
# ============================================================

with open(
    '../data/xgboost_best_thresholds.json',
    'w'
) as archivo:

    json.dump(
        BEST_THRESHOLDS,
        archivo,
        indent=4
    )


print(
    '\nThresholds guardados en: '
    '../data/xgboost_best_thresholds.json'
)


# ============================================================
# 17. GUARDAR RESULTADOS COMPLETOS
# ============================================================

df_threshold_final.to_csv(

    '../data/xgboost_threshold_results.csv',

    index=False
)


print(
    'Resultados guardados en: '
    '../data/xgboost_threshold_results.csv'
)



THRESHOLD TUNING - MX
Preprocessing: normal
Cargando: ../data/train_clean_mx.csv

F1-Macro threshold 0.50: 0.6260
Threshold óptimo: 0.500
F1-Macro óptimo: 0.6260
Mejora: +0.0000


THRESHOLD TUNING - ES
Preprocessing: lemma
Cargando: ../data/train_clean_lemma_es.csv

F1-Macro threshold 0.50: 0.7135
Threshold óptimo: 0.495
F1-Macro óptimo: 0.7157
Mejora: +0.0023


THRESHOLD TUNING - CU
Preprocessing: stem
Cargando: ../data/train_clean_stem_cu.csv

F1-Macro threshold 0.50: 0.6745
Threshold óptimo: 0.505
F1-Macro óptimo: 0.6758
Mejora: +0.0014


RESULTADOS FINALES DE THRESHOLD TUNING


,variante,preprocesamiento,threshold_base,f1_threshold_05,threshold_optimo,f1_threshold_optimo,mejora,f1_referencia_anterior
0,mx,normal,0.500,0.6260,0.500,0.6260,+0.0000,0.6259
1,es,lemma,0.500,0.7135,0.495,0.7157,+0.0023,0.7135
2,cu,stem,0.500,0.6745,0.505,0.6758,+0.0014,0.6743




TOP 10 THRESHOLDS - MX


,threshold,f1_macro,accuracy,precision_macro,recall_macro
0,0.500,0.6260,0.6669,0.6258,0.6262
1,0.505,0.6233,0.6665,0.6238,0.6227
2,0.495,0.6230,0.6615,0.6220,0.6243
3,0.480,0.6220,0.6536,0.6200,0.6268
4,0.510,0.6218,0.6674,0.6233,0.6206
5,0.475,0.6212,0.6507,0.6193,0.6274
6,0.485,0.6204,0.6544,0.6186,0.6240
7,0.490,0.6201,0.6565,0.6186,0.6224
8,0.470,0.6168,0.6444,0.6153,0.6243
9,0.515,0.6167,0.6653,0.6193,0.6149




TOP 10 THRESHOLDS - ES


,threshold,f1_macro,accuracy,precision_macro,recall_macro
0,0.495,0.7157,0.7490,0.7173,0.7143
1,0.500,0.7135,0.7477,0.7158,0.7115
2,0.490,0.7130,0.7456,0.7137,0.7124
3,0.485,0.7126,0.7435,0.7119,0.7133
4,0.480,0.7110,0.7410,0.7095,0.7127
5,0.505,0.7108,0.7465,0.7142,0.7080
6,0.475,0.7107,0.7398,0.7086,0.7133
7,0.510,0.7089,0.7460,0.7137,0.7052
8,0.470,0.7072,0.7352,0.7044,0.7111
9,0.515,0.7055,0.7444,0.7117,0.7011




TOP 10 THRESHOLDS - CU


,threshold,f1_macro,accuracy,precision_macro,recall_macro
0,0.505,0.6758,0.7342,0.7019,0.6669
1,0.500,0.6745,0.7317,0.6980,0.6659
2,0.510,0.6735,0.7338,0.7020,0.6644
3,0.495,0.6720,0.7279,0.6925,0.6641
4,0.490,0.6710,0.7258,0.6896,0.6634
5,0.515,0.6709,0.7338,0.7030,0.6616
6,0.520,0.6695,0.7346,0.7053,0.6600
7,0.485,0.6685,0.7221,0.6845,0.6616
8,0.525,0.6680,0.7350,0.7069,0.6584
9,0.480,0.6668,0.7183,0.6797,0.6606




THRESHOLDS DEFINITIVOS
{'mx': 0.5000000000000002, 'es': 0.4950000000000003, 'cu': 0.5050000000000003}

Thresholds guardados en: ../data/xgboost_best_thresholds.json
Resultados guardados en: ../data/xgboost_threshold_results.csv
